<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/01_Data_Scientist/beginner/02_pandas_numpy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pandas & NumPy — The DS Power Duo

90% of your daily DS work lives in these two libraries. This notebook goes beyond the basics — vectorization, performance, and professional patterns.

**Topics:** Advanced indexing, vectorization, merging, groupby, pivot tables, memory optimization

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
pd.set_option('display.float_format', '{:.2f}'.format)
print('NumPy:', np.__version__, '| Pandas:', pd.__version__)

## 1. Generate a Realistic Dataset

In [ ]:
# Synthetic e-commerce transaction dataset
n = 10_000
np.random.seed(42)

categories = ['Electronics', 'Clothing', 'Books', 'Food', 'Sports']
regions = ['North', 'South', 'East', 'West']

df = pd.DataFrame({
    'date': pd.date_range('2023-01-01', periods=n, freq='H'),
    'customer_id': np.random.randint(1000, 5000, n),
    'category': np.random.choice(categories, n),
    'region': np.random.choice(regions, n),
    'quantity': np.random.randint(1, 10, n),
    'price': np.random.exponential(50, n) + 10,
    'discount': np.random.choice([0, 0, 0, 0.1, 0.2, 0.3], n),
    'returned': np.random.choice([False, False, False, False, True], n),
})

df['revenue'] = df['quantity'] * df['price'] * (1 - df['discount'])

print('Shape:', df.shape)
print('\nDtypes:')
print(df.dtypes)
print('\nFirst 3 rows:')
df.head(3)

## 2. loc vs iloc — Indexing Mastery

In [ ]:
# loc: label-based indexing (row labels + column names)
# iloc: integer-based indexing (row/col positions)

# Boolean indexing
high_value = df.loc[df['revenue'] > 500, ['customer_id', 'category', 'revenue']]
print('High value transactions:', len(high_value))
print(high_value.head(3))

# Multiple conditions
electronics_north = df.loc[
    (df['category'] == 'Electronics') & 
    (df['region'] == 'North') & 
    (~df['returned']),
    ['date', 'revenue', 'quantity']
]
print(f'\nElectronics in North (not returned): {len(electronics_north)}')

# String methods
df['category_upper'] = df['category'].str.upper()
df['category_len']   = df['category'].str.len()

# datetime indexing
jan = df.loc[df['date'].dt.month == 1]
weekends = df.loc[df['date'].dt.dayofweek >= 5]
print(f'January transactions: {len(jan)}')
print(f'Weekend transactions: {len(weekends)}')

# Clean up temp columns
df.drop(columns=['category_upper', 'category_len'], inplace=True)

## 3. GroupBy — The Workhorse

In [ ]:
# Single-level groupby
category_stats = df.groupby('category').agg(
    total_revenue=('revenue', 'sum'),
    avg_revenue=('revenue', 'mean'),
    n_transactions=('revenue', 'count'),
    return_rate=('returned', 'mean'),
    avg_quantity=('quantity', 'mean')
).round(2).sort_values('total_revenue', ascending=False)

print('Category Performance:')
print(category_stats)

# Multi-level groupby
region_category = df.groupby(['region', 'category'])['revenue'].sum().unstack()
print('\nRevenue by Region x Category:')
print(region_category.round(0).astype(int))

# Transform: compute relative to group (without collapsing rows)
df['revenue_rank_in_category'] = df.groupby('category')['revenue'].rank(ascending=False)
df['pct_of_category_revenue']  = df.groupby('category')['revenue'].transform(
    lambda x: x / x.sum() * 100
)

print('\nTop 3 transactions by revenue within category:')
top3 = df[df['revenue_rank_in_category'] <= 3][['category', 'revenue', 'revenue_rank_in_category']]
print(top3.sort_values(['category', 'revenue_rank_in_category']).head(15))

## 4. Merging and Joining

In [ ]:
# Create a customer master table
unique_customers = df['customer_id'].unique()
customer_master = pd.DataFrame({
    'customer_id': unique_customers,
    'segment': np.random.choice(['Gold', 'Silver', 'Bronze'], len(unique_customers)),
    'acquisition_year': np.random.choice([2019, 2020, 2021, 2022, 2023], len(unique_customers)),
    'lifetime_value': np.random.exponential(500, len(unique_customers)),
})

# INNER JOIN: only matching customers
df_enriched = df.merge(customer_master, on='customer_id', how='inner')
print(f'After inner join: {len(df_enriched)} rows (same as original since all customers have a record)')

# LEFT JOIN: keep all transactions even without customer record
df_left = df.merge(customer_master, on='customer_id', how='left')
print(f'After left join: {len(df_left)} rows')

# Revenue by customer segment
segment_stats = df_enriched.groupby('segment').agg(
    avg_revenue=('revenue', 'mean'),
    total_revenue=('revenue', 'sum'),
    n_customers=('customer_id', 'nunique')
).round(2)
print('\nRevenue by Segment:')
print(segment_stats)

## 5. Vectorization vs Loops — Performance

In [ ]:
import time

n_rows = 100_000
arr = np.random.randn(n_rows)
df_perf = pd.DataFrame({'value': arr})

# Task: compute sigmoid of each value
def sigmoid(x): return 1 / (1 + np.exp(-x))

# Method 1: Python loop (slow)
start = time.perf_counter()
result_loop = [sigmoid(x) for x in df_perf['value']]
t_loop = time.perf_counter() - start

# Method 2: Pandas apply (moderate)
start = time.perf_counter()
result_apply = df_perf['value'].apply(sigmoid)
t_apply = time.perf_counter() - start

# Method 3: NumPy vectorized (fast)
start = time.perf_counter()
result_numpy = sigmoid(df_perf['value'].values)  # .values = numpy array
t_numpy = time.perf_counter() - start

print(f'Python loop: {t_loop:.4f}s')
print(f'Pandas apply:{t_apply:.4f}s  ({t_loop/t_apply:.1f}x faster than loop)')
print(f'NumPy:       {t_numpy:.4f}s  ({t_loop/t_numpy:.1f}x faster than loop)')
print('\n→ Rule: Always use NumPy operations on .values, not Python loops!')

## 6. Memory Optimization

In [ ]:
def mem_usage(df):
    return df.memory_usage(deep=True).sum() / 1024**2  # MB

print(f'Original memory: {mem_usage(df):.2f} MB')
print('\nDtypes before optimization:')
print(df.dtypes)

# Optimize dtypes
df_opt = df.copy()

# Categorical for low-cardinality string columns
for col in ['category', 'region']:
    df_opt[col] = df_opt[col].astype('category')

# Downcast integers
for col in ['customer_id', 'quantity']:
    df_opt[col] = pd.to_numeric(df_opt[col], downcast='integer')

# Downcast floats
for col in ['price', 'discount', 'revenue']:
    df_opt[col] = pd.to_numeric(df_opt[col], downcast='float')

print(f'\nOptimized memory: {mem_usage(df_opt):.2f} MB')
print(f'Reduction: {(1 - mem_usage(df_opt)/mem_usage(df)):.1%}')
print('\nDtypes after optimization:')
print(df_opt.dtypes)

## 7. Method Chaining — Readable Data Pipelines

In [ ]:
# Method chaining: build a pipeline in one readable expression
result = (
    df
    .query('not returned')                    # filter returned items
    .assign(                                   # add derived columns
        month=lambda x: x['date'].dt.month,
        net_revenue=lambda x: x['revenue'],
        is_discounted=lambda x: x['discount'] > 0
    )
    .groupby(['category', 'month'])           # group
    .agg(
        total_revenue=('net_revenue', 'sum'),
        avg_order=('net_revenue', 'mean'),
        discounted_pct=('is_discounted', 'mean')
    )
    .round(2)
    .sort_values('total_revenue', ascending=False)
    .head(10)
)

print('Top 10 Category-Month combinations (non-returned orders):')
print(result)

# Pivot table
pivot = (
    df.query('not returned')
    .assign(month=lambda x: x['date'].dt.month_name().str[:3])
    .pivot_table(values='revenue', index='category', 
                 columns='month', aggfunc='sum', fill_value=0)
)
print('\nPivot: Revenue by Category x Month (first 3 months):')
print(pivot.iloc[:, :3].round(0).astype(int))